<a href="https://colab.research.google.com/github/rgali32712/olist-customer-pipeline/blob/main/Customer_Data_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from google.colab import files
uploaded = files.upload()

Saving olist_customers_dataset.csv to olist_customers_dataset (2).csv
Saving olist_geolocation_dataset.csv to olist_geolocation_dataset.csv
Saving olist_order_items_dataset.csv to olist_order_items_dataset.csv
Saving olist_order_payments_dataset.csv to olist_order_payments_dataset.csv
Saving olist_order_reviews_dataset.csv to olist_order_reviews_dataset.csv
Saving olist_orders_dataset.csv to olist_orders_dataset.csv
Saving olist_products_dataset.csv to olist_products_dataset.csv
Saving olist_sellers_dataset.csv to olist_sellers_dataset.csv
Saving product_category_name_translation.csv to product_category_name_translation.csv


In [7]:
import pandas as pd

customers = pd.read_csv('olist_customers_dataset.csv')
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
payments = pd.read_csv('olist_order_payments_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
sellers = pd.read_csv('olist_sellers_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

In [8]:
for name, df in [('customers', customers), ('orders', orders),
                  ('items', items), ('payments', payments),
                  ('reviews', reviews), ('products', products),
                  ('sellers', sellers), ('category', category_translation)]:
    print(f"{name}: {df.shape}")

customers: (99441, 5)
orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
category: (71, 2)


In [9]:

print("=== ORDERS ===")
print(orders.info())
print(orders.isnull().sum())

print("\n=== PAYMENTS ===")
print(payments.info())
print(payments.isnull().sum())

print("\n=== ITEMS ===")
print(items.info())
print(items.isnull().sum())

=== ORDERS ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB
None
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivere

In [10]:

date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

orders_clean = orders.dropna(subset=['order_delivered_customer_date'])

print(f"Orders before cleaning: {len(orders)}")
print(f"Orders after cleaning: {len(orders_clean)}")

Orders before cleaning: 99441
Orders after cleaning: 96476


In [11]:

df = orders_clean.merge(customers, on='customer_id', how='left')

df = df.merge(payments, on='order_id', how='left')

df = df.merge(items, on='order_id', how='left')

print(f"Final shape: {df.shape}")
print(df.head())

Final shape: (115037, 22)
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
2  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
3  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef   
4  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf3443c3d931a367089   

  order_status order_purchase_timestamp   order_approved_at  \
0    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
1    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
2    delivered      2017-10-02 10:56:33 2017-10-02 11:07:15   
3    delivered      2018-07-24 20:41:37 2018-07-26 03:24:27   
4    delivered      2018-08-08 08:38:49 2018-08-08 08:55:23   

  order_delivered_carrier_date order_delivered_customer_date  \
0          2017-10-04 19:55:00           2017-10-10 21:25:13   
1          2017-10-04 19:55:

In [12]:

payments_agg = payments.groupby('order_id').agg(
    total_payment=('payment_value', 'sum'),
    payment_count=('payment_sequential', 'count'),
    payment_type=('payment_type', 'first')
).reset_index()

items_agg = items.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    item_count=('order_item_id', 'count')
).reset_index()

df = orders_clean.merge(customers, on='customer_id', how='left')
df = df.merge(payments_agg, on='order_id', how='left')
df = df.merge(items_agg, on='order_id', how='left')

print(f"Final shape: {df.shape}")

Final shape: (96476, 18)


In [13]:

df['delivery_days'] = (df['order_delivered_customer_date'] -
                       df['order_purchase_timestamp']).dt.days

df['order_month'] = df['order_purchase_timestamp'].dt.to_period('M')
df['order_year'] = df['order_purchase_timestamp'].dt.year

df['delivery_status'] = (df['order_delivered_customer_date'] <=
                         df['order_estimated_delivery_date']).map({True: 'On Time', False: 'Late'})

print(df[['delivery_days', 'order_month', 'delivery_status']].head(10))
print(f"\nNull check:\n{df.isnull().sum()}")

   delivery_days order_month delivery_status
0              8     2017-10         On Time
1             13     2018-07         On Time
2              9     2018-08         On Time
3             13     2017-11         On Time
4              2     2018-02         On Time
5             16     2017-07         On Time
6              9     2017-05         On Time
7              9     2017-01         On Time
8             18     2017-07         On Time
9             12     2017-05         On Time

Null check:
order_id                          0
customer_id                       0
order_status                      0
order_purchase_timestamp          0
order_approved_at                14
order_delivered_carrier_date      1
order_delivered_customer_date     0
order_estimated_delivery_date     0
customer_unique_id                0
customer_zip_code_prefix          0
customer_city                     0
customer_state                    0
total_payment                     1
payment_count           

In [14]:

df = df.dropna(subset=['total_payment'])

print(f"Final clean shape: {df.shape}")
print(f"\nBasic stats:")
print(df[['total_payment', 'delivery_days', 'item_count']].describe())

Final clean shape: (96475, 22)

Basic stats:
       total_payment  delivery_days    item_count
count   96475.000000   96475.000000  96475.000000
mean      159.853137      12.093651      1.142192
std       218.815135       9.550843      0.538785
min         9.590000       0.000000      1.000000
25%        61.880000       6.000000      1.000000
50%       105.280000      10.000000      1.000000
75%       176.330000      15.000000      1.000000
max     13664.080000     209.000000     21.000000


In [15]:
customer_summary = df.groupby('customer_unique_id').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment', 'sum'),
    avg_order_value=('total_payment', 'mean'),
    avg_delivery_days=('delivery_days', 'mean'),
    total_items=('item_count', 'sum'),
    customer_state=('customer_state', 'first'),
    customer_city=('customer_city', 'first'),
    first_order=('order_purchase_timestamp', 'min'),
    last_order=('order_purchase_timestamp', 'max'),
    on_time_rate=('delivery_status', lambda x: (x == 'On Time').mean() * 100)
).reset_index()

print(f"Customer summary shape: {customer_summary.shape}")
print(customer_summary.head())

Customer summary shape: (93355, 11)
                 customer_unique_id  total_orders  total_revenue  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1         141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1          27.19   
2  0000f46a3911fa3c0805444483337064             1          86.22   
3  0000f6ccb0745a6a4b88665a16c9f078             1          43.62   
4  0004aac84e0df4da2b147fca70cf8255             1         196.89   

   avg_order_value  avg_delivery_days  total_items customer_state  \
0           141.90                6.0            1             SP   
1            27.19                3.0            1             SP   
2            86.22               25.0            1             SC   
3            43.62               20.0            1             PA   
4           196.89               13.0            1             SP   

  customer_city         first_order          last_order  on_time_rate  
0       cajamar 2018-05-10 10:56:27 2018-05-10 10:56:27         100.

In [16]:

monthly_trends = df.groupby('order_month').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment', 'sum'),
    avg_order_value=('total_payment', 'mean')
).reset_index()

monthly_trends['order_month'] = monthly_trends['order_month'].astype(str)

state_summary = df.groupby('customer_state').agg(
    total_orders=('order_id', 'nunique'),
    total_revenue=('total_payment', 'sum'),
    avg_delivery_days=('delivery_days', 'mean'),
    on_time_rate=('delivery_status', lambda x: (x == 'On Time').mean() * 100)
).reset_index()

print("Monthly trends shape:", monthly_trends.shape)
print(monthly_trends.head())
print("\nState summary shape:", state_summary.shape)
print(state_summary.head())

Monthly trends shape: (22, 4)
  order_month  total_orders  total_revenue  avg_order_value
0     2016-10           270       47271.20       175.078519
1     2016-12             1          19.62        19.620000
2     2017-01           750      127545.67       170.060893
3     2017-02          1653      271298.65       164.125015
4     2017-03          2546      414369.39       162.753099

State summary shape: (27, 5)
  customer_state  total_orders  total_revenue  avg_delivery_days  on_time_rate
0             AC            80       19586.25          20.637500     96.250000
1             AL           397       94195.79          24.040302     76.070529
2             AM           145       27596.18          25.986207     95.862069
3             AP            67       16141.81          26.731343     95.522388
4             BA          3256      591270.60          18.866400     85.964373


In [17]:
customer_summary.to_csv('customer_summary.csv', index=False)
monthly_trends.to_csv('monthly_trends.csv', index=False)
state_summary.to_csv('state_summary.csv', index=False)

print("All 3 files exported!")

All 3 files exported!


In [23]:
from google.colab import files
files.download('customer_summary.csv')
files.download('monthly_trends.csv')
files.download('state_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!pip install snowflake-connector-python
!pip install "snowflake-connector-python[pandas]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.8 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pyopenssl to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00
  Attempting uninstall: cryptography
    Found existing installation: cryptography 43.0.3
    Uninstalling cryptography-43.0.3:
      Successfully uninstalled cryptography-43.0.3
  Attempting uninstall: pyOpenSSL
    Found 

In [20]:
import snowflake.connector

conn = snowflake.connector.connect(
    user='RGALI3',
    password='WaaGFbEB5vGmQCu',
    account='axyxcct-xn60844',
    warehouse='COMPUTE_WH',
)
cursor = conn.cursor()
cursor.execute('SELECT CURRENT_VERSION()')
print(cursor.fetchone())

('10.7.2',)


In [21]:
from snowflake.connector.pandas_tools import write_pandas

cursor.execute('CREATE DATABASE IF NOT EXISTS OLIST_DB')
cursor.execute('USE DATABASE OLIST_DB')
cursor.execute('CREATE SCHEMA IF NOT EXISTS PUBLIC')
cursor.execute('USE SCHEMA PUBLIC')
cursor.execute('USE WAREHOUSE COMPUTE_WH')

write_pandas(conn, customer_summary, 'CUSTOMER_SUMMARY', auto_create_table=True)
write_pandas(conn, monthly_trends, 'MONTHLY_TRENDS', auto_create_table=True)
write_pandas(conn, state_summary, 'STATE_SUMMARY', auto_create_table=True)

print("All 3 tables loaded into Snowflake!")

All 3 tables loaded into Snowflake!
